In [1]:
!pip install ../input/efficientnet-pytorch-070/efficientnet_pytorch-0.7.0-py3-none-any.whl

Processing /kaggle/input/efficientnet-pytorch-070/efficientnet_pytorch-0.7.0-py3-none-any.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/efficientnet-pytorch-070/efficientnet_pytorch-0.7.0-py3-none-any.whl'



In [2]:
import os

import albumentations
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms


from efficientnet_pytorch import EfficientNet

ModuleNotFoundError: No module named 'efficientnet_pytorch'

In [3]:
model_path = "../input/en-b4-tta-calr-40/model_40.pth"
sample_sub_path = "../input/cassava-leaf-disease-classification/sample_submission.csv"
test_images_path = "../input/cassava-leaf-disease-classification/test_images"

# Creating model and loading weights

In [4]:
# Loading Efficient B4 from PyTorch and update it's fc layer output to 5 classes
model = model = EfficientNet.from_name('efficientnet-b4', num_classes=5)

model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model.load_state_dict(torch.load(model_path))
model.eval()

NameError: name 'EfficientNet' is not defined

# Submission Augmentations

In [5]:
sub_aug = albumentations.Compose([
                albumentations.RandomResizedCrop(256, 256, scale=(0.5, 1.0)),
                albumentations.Transpose(p=0.8),
                albumentations.HorizontalFlip(p=0.5),
                albumentations.VerticalFlip(p=0.5),
                albumentations.ShiftScaleRotate(p=0.8),
                albumentations.HueSaturationValue(
                    hue_shift_limit=20, 
                    sat_shift_limit=30, 
                    val_shift_limit=0, 
                    p=0.5
                ),
                albumentations.RandomBrightnessContrast(
                    brightness_limit=(-0.1,0.1), 
                    contrast_limit=(-0.1, 0.1), 
                    p=0.5
                ),
                albumentations.Normalize(
                    mean=[0.485, 0.456, 0.406], 
                    std=[0.229, 0.224, 0.225], 
                    max_pixel_value=255.0, 
                    p=1.0
                ),
                albumentations.CoarseDropout(max_holes=20, max_height=10, max_width=10, p=0.5),
                albumentations.Cutout(num_holes=10, max_h_size=10, max_w_size=10, p=0.5)
            ], p=1.)

ValueError: 1 validation error for InitSchema
size
  Input should be a valid tuple [type=tuple_type, input_value=256, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/tuple_type

# Creating the Submission file

In [6]:
sample_sub = pd.read_csv(sample_sub_path)

predictions = []
for _, sample_row in sample_sub.iterrows():
    
    image_pred = 0
    for j in range(5):
        image = np.array(Image.open(os.path.join(test_images_path, sample_row.image_id)))
        image = sub_aug(image=image)["image"]
        image = transforms.ToTensor()(np.array(image))
        image = image.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        outputs = model(image.unsqueeze(0))
        
        image_pred += outputs
    image_pred /= 10
    _, pred_label = torch.max(image_pred, 1)
        
    predictions.append([sample_row.image_id, pred_label.item()])

sub_df = pd.DataFrame(predictions,columns=['image_id', 'label'])
sub_df.to_csv('submission.csv', index=False)
print(sub_df.head())


NameError: name 'sub_aug' is not defined